# Beard Flux — Improved Color-correction and Masking

This notebook provides a robust pipeline to: detect/refine a beard mask, inpaint the beard area to create a clean face, align a reference silhouette, generate or accept a generated beard texture restricted to that silhouette, perform color-matching (chroma-only) confined to the beard mask, and composite back using seamless cloning or alpha blending.

The key fixes implemented here (compared to the previous `beard_flux_fixed.ipynb`) are:
- All color correction steps run exclusively inside the beard mask to avoid background artifacts.
- Color transfer uses LAB chroma mean-shift (a/b channels only) to preserve luminance/texture.
- Semantic subregion isolation (e.g. mustache band) is provided to avoid producing full beards when a mustache-only reference is used.

Usage: run the cells, then call the example usage cell with your own images/masks. Replace placeholder paths with actual files.


In [ ]:
# Imports
import cv2
import numpy as np
from typing import Tuple, Dict


In [ ]:
# Utilities: refine mask, inpaint, LAB chroma match, composite, semantic isolate
def refine_mask(mask: np.ndarray, min_area=300, blur_sigma=5) -> np.ndarray:
    """Refine a binary mask (0..255 or 0..1) into a feathered alpha (0..1).
    - removes small components, opens, and Gaussian-blurs to produce alpha.
    """
    if mask.dtype != np.uint8:
        mask_u8 = (mask * 255).astype(np.uint8)
    else:
        mask_u8 = mask.copy()
    # threshold to binary
    _, bw = cv2.threshold(mask_u8, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(bw)
    for c in contours:
        if cv2.contourArea(c) >= min_area:
            cv2.drawContours(out, [c], -1, 255, -1)
    # morphological open to remove noise
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    out = cv2.morphologyEx(out, cv2.MORPH_OPEN, k)
    # feather to alpha
    alpha = cv2.GaussianBlur(out.astype(np.float32)/255.0, (0,0), blur_sigma)
    alpha = np.clip(alpha, 0.0, 1.0)
    return alpha

def inpaint_beard(img_bgr: np.ndarray, beard_alpha: np.ndarray) -> np.ndarray:
    """Inpaint the beard region (beard_alpha in 0..1) to produce a clean face.
    Uses OpenCV Telea inpaint as a simple fallback.
    """
    mask_uint8 = (beard_alpha > 0.5).astype(np.uint8) * 255
    inpainted = cv2.inpaint(img_bgr, mask_uint8, 3, cv2.INPAINT_TELEA)
    return inpainted

def match_beard_color(gen_bgr: np.ndarray, target_bgr: np.ndarray, mask_alpha: np.ndarray, use_target_mask_alpha: np.ndarray = None) -> np.ndarray:
    """Match chroma (a,b) channels of gen to target inside mask.
    - gen_bgr: generated beard image (full image or same size).
    - target_bgr: source image with desired beard color (original face).
    - mask_alpha: float alpha 0..1 specifying pixels to modify.
    - use_target_mask_alpha: optionally a mask to compute target color means (if different region).
    Returns BGR image where only masked pixels' a/b were shifted.
    """
    # Convert to LAB float32 for arithmetic
    gen_lab = cv2.cvtColor(gen_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    tgt_lab = cv2.cvtColor(target_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    mask = (mask_alpha > 0.5)
    if use_target_mask_alpha is not None:
        tgt_mask = (use_target_mask_alpha > 0.5)
    else:
        tgt_mask = mask
    # guard against empty masks
    if not np.any(mask):
        return gen_bgr
    # compute mean a,b where masks indicate
    gen_a = gen_lab[:,:,1][mask]
    gen_b = gen_lab[:,:,2][mask]
    tgt_a = tgt_lab[:,:,1][tgt_mask] if np.any(tgt_mask) else gen_a
    tgt_b = tgt_lab[:,:,2][tgt_mask] if np.any(tgt_mask) else gen_b
    gen_a_mean = float(np.mean(gen_a))
    gen_b_mean = float(np.mean(gen_b))
    tgt_a_mean = float(np.mean(tgt_a))
    tgt_b_mean = float(np.mean(tgt_b))
    delta_a = tgt_a_mean - gen_a_mean
    delta_b = tgt_b_mean - gen_b_mean
    # apply deltas only to mask pixels (keep L channel unchanged)
    # Work on float array then clip
    gen_lab[:,:,1][mask] = np.clip(gen_lab[:,:,1][mask] + delta_a, 0, 255)
    gen_lab[:,:,2][mask] = np.clip(gen_lab[:,:,2][mask] + delta_b, 0, 255)
    out_bgr = cv2.cvtColor(gen_lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
    return out_bgr

def composite_beard(base_bgr: np.ndarray, beard_bgr: np.ndarray, beard_alpha: np.ndarray, method='seamless') -> np.ndarray:
    """Composite the beard over base using either seamless cloning or alpha blend.
    beard_alpha: 0..1 float alpha
    method: 'seamless' or 'alpha'
    """
    h,w = beard_alpha.shape[:2]
    mask_uint8 = (beard_alpha * 255).astype(np.uint8)
    if method == 'seamless':
        ys, xs = np.where(mask_uint8>0)
        if ys.size == 0:
            return base_bgr
        center = (int(xs.mean()), int(ys.mean()))
        # seamlessClone requires 8-bit 3-channel images of same size
        cloned = cv2.seamlessClone(beard_bgr, base_bgr, mask_uint8, center, cv2.NORMAL_CLONE)
        return cloned
    else:
        alpha_3 = beard_alpha[...,None].astype(np.float32)
        comp = (alpha_3 * beard_bgr.astype(np.float32) + (1-alpha_3) * base_bgr.astype(np.float32)).astype(np.uint8)
        return comp

def isolate_mustache(silhouette_mask: np.ndarray, landmarks: Dict[str,Tuple[int,int]]) -> np.ndarray:
    """Isolate a mustache subregion from a silhouette using nose/lip landmarks.
    silhouette_mask: binary 0/255 or 0..1 mask same size as image.
    landmarks: dict with 'nose_tip','upper_lip','lower_lip' keys mapping to (x,y)
    Returns refined feathered mustache alpha (0..1).
    """
    if silhouette_mask.dtype != np.uint8:
        s = (silhouette_mask * 255).astype(np.uint8)
    else:
        s = silhouette_mask.copy()
    h, w = s.shape[:2]
    nose_y = landmarks['nose_tip'][1]
    upper_lip_y = landmarks['upper_lip'][1]
    lower_lip_y = landmarks['lower_lip'][1]
    band_top = int(max(0, nose_y + 0.15*(upper_lip_y - nose_y)))
    band_bottom = int(min(h-1, upper_lip_y + 0.2*(lower_lip_y - upper_lip_y)))
    band_mask = np.zeros_like(s, dtype=np.uint8)
    band_mask[band_top:band_bottom, :] = 255
    mustache = cv2.bitwise_and(s, band_mask)
    # refine and feather
    return refine_mask(mustache, min_area=50, blur_sigma=3)


In [ ]:
# Example usage (replace placeholders with your images/masks)
# NOTE: this cell is a template — adjust file paths and model generation steps accordingly.

# load images (BGR)
# original_face = cv2.imread('input_face.jpg')
# reference_silhouette = cv2.imread('ref_silhouette.png', cv2.IMREAD_UNCHANGED)  # use alpha or grayscale silhouette
# original_beard_mask = cv2.imread('orig_beard_mask.png', cv2.IMREAD_GRAYSCALE)

# For demo, we skip generative step — assume `generated_beard_texture` is your model output image that contains beard texture.
# Step 1: refine original beard mask
# orig_alpha = refine_mask(original_beard_mask)
# Step 2: inpaint to produce clean face for conditioning
# clean_face = inpaint_beard(original_face, orig_alpha)
# Step 3: align reference silhouette to target (not implemented here — depends on landmarks & your warp)
# aligned_silhouette_mask = ... (binary mask)
# aligned_alpha = refine_mask(aligned_silhouette_mask)
# Step 4: run your generative model conditioned on aligned_alpha; obtain generated_beard_texture (BGR)
# Step 5: color-match generated beard to original beard color (use orig_alpha or hair sampling)
# matched_beard = match_beard_color(generated_beard_texture, original_face, aligned_alpha, use_target_mask_alpha=orig_alpha)
# Step 6: composite back onto clean_face
# final = composite_beard(clean_face, matched_beard, aligned_alpha, method='seamless')
# cv2.imwrite('final_beard_result.jpg', final)
print('Template usage: replace placeholders and run the cells. Functions for mask refinement, color matching, and compositing are provided.')


Notes and tips:
- Use LAB mean-shift instead of whole-image histogram-matching. This prevents luminance/texture change and keeps adjustments localized.
- Always inspect the masks visually (display alpha map) to ensure no stray non-zero values outside the beard region.
- For mustache-only references, compute a mustache sub-mask (function provided) and use it instead of the full silhouette.
- If you want, I can open a PR updating the original `beard_flux_fixed.ipynb` in-place instead of adding this new file.
